In [2]:
from pybaseball import statcast
import pandas as pd
import datetime
import warnings
warnings.filterwarnings("ignore")

In [74]:
df = statcast(start_dt='2026-03-25', end_dt='2026-10-10')

This is a large query, it may take a moment to complete


 49%|████▉     | 98/200 [01:21<00:05, 17.73it/s]/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
 82%|████████▎ | 165/200 [01:24<00:01, 19.91it/s]/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
100%|██████████| 200/200 [01:26<00:00,  2.32it/s]


In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 264436 entries, 1766 to 251
Columns: 118 entries, pitch_type to intercept_ball_minus_batter_pos_y_inches
dtypes: Float64(42), Int64(59), datetime64[ns](1), object(16)
memory usage: 265.6+ MB


In [77]:
dt = df[df['events'].isin(['field_out', 'force_out', 'single', 'double', 'strikeout', 'home_run', 'grounded_into_double_play', 'triple', 'fielders_choice_out', 'double_play', 'field_error', 'fielders_choice', 'strikeout_double_play', 'walk', 'hit_by_pitch']) == True][['game_date', 'batter', 'events', 'p_throws', 'home_team', 'away_team']]
dt = dt[['game_date', 'batter', 'events', 'p_throws']]
dt.head()

,game_date,batter,events,p_throws
1766,2026-06-01,669257,field_out,R
2075,2026-06-01,663656,field_out,R
2459,2026-06-01,605141,field_out,R
1790,2026-06-01,682998,field_out,L
1902,2026-06-01,606466,home_run,L


## WAVE

In [120]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=81)

sev = dt[dt['game_date'] >= sev_days_ago]
def  hit(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 1
    elif df['events'] == 'triple':
        return 1
    elif df['events'] == 'home_run':
        return 1
    else:
        return 0
sev['hit'] = sev.apply(hit, axis=1)
lsev = sev[sev['p_throws'] == "L"].drop('p_throws', axis=1)
lsev['at_bat_lsev'] = 1
lsev = lsev[['batter', 'hit', 'at_bat_lsev']].groupby('batter', as_index = False).sum()
lsev['lsev_avg'] = lsev['hit'] / lsev['at_bat_lsev']
lsev = lsev[['batter', 'lsev_avg', 'at_bat_lsev']]
lsev.head()

,batter,lsev_avg,at_bat_lsev
0,457705,0.160000,50
1,467793,0.250000,4
2,500743,0.264151,53
3,502054,0.000000,11
4,502671,0.288136,59


In [121]:
rsev = sev[sev['p_throws'] == "R"].drop('p_throws', axis=1)
rsev['at_bat_rsev'] = 1
rsev = rsev[['batter', 'hit', 'at_bat_rsev']].groupby('batter', as_index = False).sum()
rsev['rsev_avg'] = rsev['hit'] / rsev['at_bat_rsev']
rsev = rsev[['batter', 'rsev_avg', 'at_bat_rsev']]
rsev.head()

,batter,rsev_avg,at_bat_rsev
0,457705,0.187500,32
1,467793,0.045455,22
2,500743,0.209302,43
3,502054,0.000000,3
4,502671,0.176471,68


In [122]:
fifteen_days_ago = latest - datetime.timedelta(days=10)

fif = dt[dt['game_date'] >= fifteen_days_ago]
fif['hit'] = fif.apply(hit, axis=1)
lfif = fif[fif['p_throws'] == "L"].drop('p_throws', axis=1)
lfif['at_bat_lfif'] = 1
lfif = lfif[['batter', 'hit', 'at_bat_lfif']].groupby('batter', as_index = False).sum()
lfif['lfif_avg'] = lfif['hit'] / lfif['at_bat_lfif']
lfif = lfif[['batter', 'lfif_avg', 'at_bat_lfif']]
lfif.head()

,batter,lfif_avg,at_bat_lfif
0,457705,0.000000,5
1,500743,0.153846,13
2,502671,0.444444,9
3,506702,0.000000,6
4,516782,0.000000,6


In [123]:
rfif = fif[fif['p_throws'] == "R"].drop('p_throws', axis=1)
rfif['at_bat_rfif'] = 1
rfif = rfif[['batter', 'hit', 'at_bat_rfif']].groupby('batter', as_index = False).sum()
rfif['rfif_avg'] = rfif['hit'] / rfif['at_bat_rfif']
rfif = rfif[['batter', 'rfif_avg', 'at_bat_rfif']]
rfif.head()

,batter,rfif_avg,at_bat_rfif
0,457705,0.000000,5
1,500743,0.111111,9
2,502671,0.148148,27
3,506702,0.000000,4
4,516782,0.000000,1


In [124]:
thirty_days_ago = latest - datetime.timedelta(days=30)

thir = dt[dt['game_date'] >= thirty_days_ago]
thir['hit'] = thir.apply(hit, axis=1)
lthir = thir[thir['p_throws'] == "L"].drop('p_throws', axis=1)
lthir['at_bat_lthir'] = 1
lthir = lthir[['batter', 'hit', 'at_bat_lthir']].groupby('batter', as_index = False).sum()
lthir['lthir_avg'] = lthir['hit'] / lthir['at_bat_lthir']
lthir = lthir[['batter', 'lthir_avg', 'at_bat_lthir']]
lthir.head()

,batter,lthir_avg,at_bat_lthir
0,457705,0.090909,22
1,500743,0.160000,25
2,502671,0.454545,33
3,506702,0.133333,15
4,514888,0.153846,13


In [125]:
rthir = thir[thir['p_throws'] == "R"].drop('p_throws', axis=1)
rthir['at_bat_rthir'] = 1
rthir = rthir[['batter', 'hit', 'at_bat_rthir']].groupby('batter', as_index = False).sum()
rthir['rthir_avg'] = rthir['hit'] / rthir['at_bat_rthir']
rthir = rthir[['batter', 'rthir_avg', 'at_bat_rthir']]
rthir.head()

,batter,rthir_avg,at_bat_rthir
0,457705,0.250000,16
1,500743,0.200000,20
2,502671,0.160714,56
3,506702,0.000000,10
4,514888,0.243243,37


In [126]:
full = dt
full['hit'] = full.apply(hit, axis=1)
lfull = full[full['p_throws'] == "L"].drop('p_throws', axis=1)
lfull['l_at_bat'] = 1
lfull = lfull[['batter', 'hit', 'l_at_bat']].groupby('batter', as_index = False).sum()
lfull['lfull_avg'] = lfull['hit'] / lfull['l_at_bat']
lfull = lfull[['batter', 'lfull_avg', 'l_at_bat']]
lfull.head()

,batter,lfull_avg,l_at_bat
0,457705,0.160000,50
1,467793,0.250000,4
2,500743,0.264151,53
3,502054,0.000000,11
4,502671,0.288136,59


In [127]:
rfull = full[full['p_throws'] == "R"].drop('p_throws', axis=1)
rfull['r_at_bat'] = 1
rfull = rfull[['batter', 'hit', 'r_at_bat']].groupby('batter', as_index = False).sum()
rfull['rfull_avg'] = rfull['hit'] / rfull['r_at_bat']
rfull = rfull[['batter', 'rfull_avg', 'r_at_bat']]
rfull.head()

,batter,rfull_avg,r_at_bat
0,457705,0.187500,32
1,467793,0.045455,22
2,500743,0.209302,43
3,502054,0.000000,3
4,502671,0.176471,68


In [128]:
wave = rfull.merge(lsev, on='batter', how='left').merge(lthir, on='batter', how='left').merge(lfif, on='batter', how='left').merge(lfull, on='batter', how='left').merge(rsev, on='batter', how='left').merge(rthir, on='batter', how='left').merge(rfif, on='batter', how='left')
from pybaseball import chadwick_register
names = chadwick_register()

wave =wave.rename(columns = {'batter' : 'key_mlbam'}).merge(names, on='key_mlbam', how='left')
wave = wave.drop(['key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last'], axis=1)

wave.head()

Gathering player lookup table. This may take a moment.


,key_mlbam,rfull_avg,r_at_bat,lsev_avg,at_bat_lsev,lthir_avg,at_bat_lthir,lfif_avg,at_bat_lfif,lfull_avg,l_at_bat,rsev_avg,at_bat_rsev,rthir_avg,at_bat_rthir,rfif_avg,at_bat_rfif,name_last,name_first
0,457705,0.187500,32,0.160000,50.0,0.090909,22.0,0.000000,5.0,0.160000,50.0,0.187500,32,0.250000,16.0,0.000000,5.0,McCutchen,Andrew
1,467793,0.045455,22,0.250000,4.0,NaN,NaN,NaN,NaN,0.250000,4.0,0.045455,22,NaN,NaN,NaN,NaN,Santana,Carlos
2,500743,0.209302,43,0.264151,53.0,0.160000,25.0,0.153846,13.0,0.264151,53.0,0.209302,43,0.200000,20.0,0.111111,9.0,Rojas,Miguel
3,502054,0.000000,3,0.000000,11.0,NaN,NaN,NaN,NaN,0.000000,11.0,0.000000,3,NaN,NaN,NaN,NaN,Pham,Tommy
4,502671,0.176471,68,0.288136,59.0,0.454545,33.0,0.444444,9.0,0.288136,59.0,0.176471,68,0.160714,56.0,0.148148,27.0,Goldschmidt,Paul


In [129]:
wave["abt"] = wave['l_at_bat'] + wave['r_at_bat']
wave['abtl'] = wave['l_at_bat'] / wave['abt']
wave['abtr'] = wave['r_at_bat'] / wave['abt']
wave['WAVE_R'] = wave['rfull_avg']*.15 + wave['rsev_avg']*.25 + wave['rthir_avg'] *.275 + wave['rfif_avg'] *.325
wave['WAVE_L'] = wave['lfull_avg'] *.15 + wave['lsev_avg'] *.25 + wave['lthir_avg'] *.275 + wave['lfif_avg'] *.325
wave['WAVE'] = wave['WAVE_R']*wave['abtr'] + wave['WAVE_L']*wave['abtl']
WAVE = wave[['name_first', 'name_last', 'WAVE', 'WAVE_L', 'WAVE_R', 'l_at_bat', 'r_at_bat', 'lsev_avg', 'rsev_avg', 'key_mlbam']]
WAVE['probability'] = 1 - ((1-WAVE['WAVE'])**3.5)
WAVE['probability_L'] = 1 - ((1-WAVE['WAVE_L'])**3.5)
WAVE['probability_R'] = 1 - ((1-WAVE['WAVE_R'])**3.5)
WAVE[(WAVE['l_at_bat'] + WAVE['r_at_bat']) > 30].nlargest(30, 'probability')

,name_first,name_last,WAVE,WAVE_L,WAVE_R,l_at_bat,r_at_bat,lsev_avg,rsev_avg,key_mlbam,probability,probability_L,probability_R
524,Jung Hoo,Lee,0.399044,0.297171,0.438848,59.0,151,0.254237,0.298013,808982,0.831752,0.708946,0.867632
180,Nathan,Lukes,0.355515,0.000000,0.374476,4.0,75,0.000000,0.293333,664770,0.785096,0.000000,0.806423
384,Andrew,Vaughn,0.346468,0.435989,0.290766,28.0,45,0.392857,0.266667,683734,0.774350,0.865257,0.699555
99,Amed,Rosario,0.336565,0.345342,0.326591,50.0,44,0.240000,0.272727,642708,0.762155,0.772987,0.749403
320,Blaze,Alexander,0.327968,0.271970,0.359223,48.0,86,0.229167,0.267442,677942,0.751192,0.670752,0.789393
278,Tristan,Peters,0.320659,0.000000,0.337536,8.0,152,0.000000,0.282895,671976,0.741591,0.000000,0.763370
301,Jonny,DeLuca,0.315753,0.476296,0.191905,54.0,70,0.240741,0.271429,676356,0.735000,0.896056,0.525628
353,Mickey,Gasper,0.314063,0.150000,0.346875,9.0,45,0.222222,0.333333,681508,0.732702,0.433805,0.774842
73,Jorge,Mateo,0.312857,0.375374,0.270345,34.0,50,0.294118,0.300000,622761,0.731054,0.807394,0.668173
105,Isiah,Kiner-Falefa,0.309355,0.206151,0.375406,32.0,50,0.187500,0.340000,643396,0.726226,0.554259,0.807429


In [130]:
WAVE[WAVE['name_first'] == "Brendan"].head()

,name_first,name_last,WAVE,WAVE_L,WAVE_R,l_at_bat,r_at_bat,lsev_avg,rsev_avg,key_mlbam,probability,probability_L,probability_R
346,Brendan,Donovan,NaN,NaN,NaN,24.0,76,0.166667,0.25,680977,NaN,NaN,NaN


## P_WAVE

In [17]:
pdf = df[df['events'].isin(['field_out', 'force_out', 'single', 'double', 'strikeout', 'home_run', 'grounded_into_double_play', 'triple', 'fielders_choice_out', 'double_play', 'field_error', 'fielders_choice', 'strikeout_double_play', 'walk', 'hit_by_pitch']) == True][['game_date', 'pitcher', 'events']]
pdf.head()

,game_date,pitcher,events
1766,2026-06-01,623149,field_out
2075,2026-06-01,623149,field_out
2459,2026-06-01,623149,field_out
1790,2026-06-01,676263,field_out
1902,2026-06-01,676263,home_run


In [18]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=15)

sev = pdf[pdf['game_date'] >= sev_days_ago]
def  hit(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 1
    elif df['events'] == 'triple':
        return 1
    elif df['events'] == 'home_run':
        return 1
    else:
        return 0

def  bases(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 2
    elif df['events'] == 'triple':
        return 3
    elif df['events'] == 'home_run':
        return 4
    else:
        return 0

def stk(df):
  if df['events'] == 'strikeout':
    return 1
  elif df['events'] == 'strikeout_double_play':
    return 1
  elif df['events'] == 'walk':
    return 1
  elif df['events'] == 'hit_by_pitch':
    return 1
  else:
    return 0

def out(df):
  if df['events'] == 'single':
    return 0
  elif df['events'] == 'double':
    return 0
  elif df['events'] == 'triple':
    return 0
  elif df['events'] == 'home_run':
    return 0
  elif df['events'] == 'walk':
    return 0
  elif df['events'] == 'hit_by_pitch':
    return 0
  else:
    return 1

def homer(df):
  if df['events'] == 'home_run':
    return 1
  else:
    return 0

sev['out'] = sev.apply(out, axis=1)

sev['hit'] = sev.apply(hit, axis=1)
sev['stk'] = sev.apply(stk, axis=1)
sev['TBA'] = sev.apply(bases, axis=1)
sev['hr'] = sev.apply(homer, axis=1)
sev['at_bat_sev'] = 1
sev = sev[['pitcher', 'hit', 'stk', 'TBA', 'hr','at_bat_sev']].groupby('pitcher', as_index = False).sum()
sev['baa_fif'] = sev['hit'] / sev['at_bat_sev']
sev['stk_per_sev'] = sev['stk'] / sev['at_bat_sev']
sev['pave_fif'] = sev['baa_fif']/(1-sev['stk_per_sev'])
sev['power_a_fif'] = sev['TBA']/sev['at_bat_sev']
sev['hr_fif'] = sev['hr'] / sev['at_bat_sev']
sev = sev[['pitcher', 'pave_fif', 'baa_fif', 'power_a_fif', 'hr_fif']]
sev.head()

,pitcher,pave_fif,baa_fif,power_a_fif,hr_fif
0,445276,0.125000,0.076923,0.307692,0.076923
1,455119,0.388889,0.350000,0.750000,0.100000
2,471911,0.277778,0.238095,0.380952,0.047619
3,472610,0.500000,0.333333,0.333333,0.000000
4,489446,0.214286,0.125000,0.291667,0.041667


In [19]:
thir_days_ago = latest - datetime.timedelta(days=30)
thir = pdf[pdf['game_date'] >= thir_days_ago]
thir['hit'] = thir.apply(hit, axis=1)
thir['stk'] = thir.apply(stk, axis=1)
thir['TBA'] = thir.apply(bases, axis=1)
thir['hr'] = thir.apply(homer, axis=1)
thir['at_bat_thir'] = 1
thir = thir[['pitcher', 'hit', 'stk', 'TBA', 'hr', 'at_bat_thir']].groupby('pitcher', as_index = False).sum()
thir['baa_thir'] = thir['hit'] / thir['at_bat_thir']
thir['stk_per_thir'] = thir['stk'] / thir['at_bat_thir']
thir['pave_thir'] = thir['baa_thir']/(1-thir['stk_per_thir'])
thir['power_a_thir'] = thir['TBA']/thir['at_bat_thir']
thir['hr_thir'] = thir['hr'] / thir['at_bat_thir']
thir = thir[['pitcher', 'pave_thir', 'baa_thir', 'hr_thir', 'power_a_thir']]
thir.head()

,pitcher,pave_thir,baa_thir,hr_thir,power_a_thir
0,445276,0.076923,0.035714,0.035714,0.142857
1,455119,0.388889,0.350000,0.100000,0.750000
2,471911,0.272727,0.230769,0.038462,0.346154
3,472610,0.384615,0.312500,0.062500,0.500000
4,489446,0.285714,0.166667,0.027778,0.305556


In [20]:
half_days_ago = latest - datetime.timedelta(days=81)
half = pdf[pdf['game_date'] >= half_days_ago]
half['hit'] = half.apply(hit, axis=1)
half['stk'] = half.apply(stk, axis=1)
half['TBA'] = half.apply(bases, axis=1)
half['hr'] = half.apply(homer, axis=1)
half['at_bat_half'] = 1
half = half[['pitcher', 'hit', 'stk', 'TBA', 'hr', 'at_bat_half']].groupby('pitcher', as_index = False).sum()
half['baa_half'] = half['hit'] / half['at_bat_half']
half['stk_per_half'] = half['stk'] / half['at_bat_half']
half['pave_half'] = half['baa_half']/(1-half['stk_per_half'])
half['power_a_half'] = half['TBA']/half['at_bat_half']
half['hr_half'] = half['hr'] / half['at_bat_half']
half = half[['pitcher', 'pave_half', 'baa_half', 'power_a_half', 'hr_half']]
half.head()

,pitcher,pave_half,baa_half,power_a_half,hr_half
0,434378,0.375000,0.315789,0.631579,0.052632
1,445276,0.264706,0.145161,0.338710,0.064516
2,453286,0.343750,0.265060,0.542169,0.084337
3,455119,0.461538,0.367347,0.673469,0.061224
4,471911,0.250000,0.206897,0.310345,0.034483


In [21]:
pull = pdf
pull['hits'] = pull.apply(hit, axis=1)
pull['stk'] = pull.apply(stk, axis=1)
pull['TBA'] = pull.apply(bases, axis=1)
pull['hr'] = pull.apply(homer, axis=1)
pull['at_bats'] = 1
pull = pull[['pitcher', 'hits', 'stk', 'TBA', 'at_bats', 'hr']].groupby('pitcher', as_index = False).sum()
pull['baa_full'] = pull['hits'] / pull['at_bats']
pull['stk_per_full'] = pull['stk'] / pull['at_bats']
pull['pave_full'] = pull['baa_full']/(1-pull['stk_per_full'])
pull['power_a_full'] = pull['TBA']/pull['at_bats']
pull['hr_full'] = pull['hr'] / pull['at_bats']
pull = pull[['pitcher', 'pave_full', 'baa_full', 'power_a_full', 'hr_full', 'at_bats', 'hits', 'TBA']]
pull.head()

,pitcher,pave_full,baa_full,power_a_full,hr_full,at_bats,hits,TBA
0,434378,0.375000,0.315789,0.631579,0.052632,19,6,12
1,445276,0.264706,0.145161,0.338710,0.064516,62,9,21
2,453286,0.343750,0.265060,0.542169,0.084337,83,22,45
3,455119,0.461538,0.367347,0.673469,0.061224,49,18,33
4,471911,0.250000,0.206897,0.310345,0.034483,29,6,9


In [22]:
pave = pull.merge(thir, on='pitcher', how='left').merge(sev, on='pitcher', how='left').merge(half, on = 'pitcher', how = 'left')
pave['PAVE'] = pave['pave_full'].fillna(0) * .3 + pave['pave_thir'].fillna(0) * .265 + pave['pave_half'].fillna(0)*.23 +pave['pave_fif'].fillna(0) * .205
pave['power_a'] = pave['power_a_full'].fillna(0) * .3 + pave['power_a_thir'].fillna(0) * .265 + pave['power_a_half'].fillna(0) * .23 + pave['power_a_fif'].fillna(0) * .205
pave['baa'] = pave['baa_full'].fillna(0) * .3 + pave['baa_thir'].fillna(0) * .265 + pave['baa_half'].fillna(0) * .23 + pave['baa_fif'].fillna(0) * .205
pave['hr_per'] = pave['hr_full'].fillna(0) * .3 + pave['hr_thir'].fillna(0) * .265 + pave['hr_half'].fillna(0) * .23 + pave['hr_fif'].fillna(0) * .205

from pybaseball import chadwick_register
names = chadwick_register()

pave =pave.rename(columns = {'pitcher' : 'key_mlbam'}).merge(names, on='key_mlbam', how='left')
pave = pave.drop(['key_mlbam', 'key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last'], axis=1)

pave = pave[['name_first', 'name_last', 'PAVE', 'at_bats', 'hits', 'TBA', 'pave_fif', 'pave_thir', 'pave_half', 'pave_full', 'power_a', 'baa', 'hr_per']]

Gathering player lookup table. This may take a moment.


In [23]:
qual = pave['at_bats'].max() * .75
mean_p = pave[pave['at_bats'] > qual]['PAVE'].mean()
pave['PAVE_PLUS'] = pave['PAVE']/mean_p
pave['PAVE_PLUS'] = pave['PAVE_PLUS'].fillna(0)
pave = pave[pave['PAVE_PLUS'] > 0]
quat = pave['at_bats'].max() * .25
filt = pave[pave['at_bats'] > quat]['at_bats'].mean()
pave = pave[pave['at_bats'] > filt]
pave['Expected_Hits'] = pave['baa'] * 22
pave['Expected_Bases'] = pave['power_a'] * 22
pave['Expected_HRs'] = pave['hr_per'] * 22
pave = pave[['name_first', 'name_last', 'at_bats', 'PAVE_PLUS', 'Expected_Hits', 'Expected_Bases', 'Expected_HRs']]
pave = pave.sort_values('PAVE_PLUS', ascending=False)
pave[pave['at_bats'] > 30].nlargest(20, 'Expected_Bases')

,name_first,name_last,at_bats,PAVE_PLUS,Expected_Hits,Expected_Bases,Expected_HRs
240,Brady,Singer,236,1.201521,6.235975,12.980621,1.952056
170,Aaron,Civale,241,1.138626,6.315752,12.744847,1.674220
84,Kyle,Freeland,228,1.331207,6.913418,12.540975,1.500774
22,Michael,Lorenzen,275,1.389548,7.278097,11.366238,0.840368
80,Erick,Fedde,232,1.193318,6.216784,11.263762,1.348420
321,Trevor,Rogers,218,1.115558,5.978102,11.237599,0.974997
8,Jose,Quintana,174,1.158549,6.576778,11.232710,0.614579
44,Jameson,Taillon,253,1.074597,5.507244,11.230463,1.699081
146,Bailey,Ober,279,1.030547,5.820585,11.198098,1.334800
493,Shota,Imanaga,283,1.010249,5.035812,10.341875,1.415958


In [24]:
pave[pave['name_last'] == 'Smith'].head()

,name_first,name_last,at_bats,PAVE_PLUS,Expected_Hits,Expected_Bases,Expected_HRs


# Hit_Prob

In [131]:
gam_id = df[["game_date", "home_team", "away_team", "pitcher", "at_bat_number"]].drop_duplicates().iloc[::-1]

def create_id(df, col_check):
  count = 0
  new_col = []
  for val in df[col_check]:
      if val == 1:
          count += 1
      new_col.append(count)
  df['game_id'] = new_col
  return df

gam_id = create_id(gam_id, 'at_bat_number')

data = df.merge(gam_id, on = ["game_date", "home_team", "away_team", "pitcher", "at_bat_number"])
data['ind'] = (data['game_id'].astype('str') + data['at_bat_number'].astype('str') + data['pitch_number'].astype('str')).astype('int')
data = data.set_index('ind')
data = data.sort_index()
data[data['game_id'] == 8].head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,game_id
ind,,,,,,,,,,,,,,,,,,,,,
811,FS,2026-03-26,91.0,-1.73,5.54,"Yamamoto, Yoshinobu",606466,808967,NaN,swinging_strike,...,2.8,1.11,-1.11,42.2,16.714818,-27.679405,34.961981,32.044134,45.488505,8
812,FF,2026-03-26,96.1,-1.92,5.43,"Yamamoto, Yoshinobu",606466,808967,NaN,swinging_strike,...,1.26,0.91,-0.91,37.5,4.352523,-4.138665,34.663344,21.73934,28.812189,8
813,FS,2026-03-26,92.6,-2.01,5.34,"Yamamoto, Yoshinobu",606466,808967,NaN,ball,...,2.29,1.28,-1.28,36.2,<NA>,<NA>,<NA>,<NA>,<NA>,8
814,FS,2026-03-26,91.0,-1.95,5.5,"Yamamoto, Yoshinobu",606466,808967,NaN,foul,...,2.55,1.18,-1.18,39.4,13.7044,-8.002634,27.901996,41.100783,40.310685,8
815,CU,2026-03-26,77.8,-1.67,5.59,"Yamamoto, Yoshinobu",606466,808967,NaN,foul,...,5.15,-1.25,1.25,48.8,20.264665,-51.364537,32.465386,19.398139,60.594119,8


In [132]:
dr = data[data['events'].isin(['field_out', 'force_out', 'single', 'double', 'strikeout', 'home_run', 'grounded_into_double_play', 'triple', 'fielders_choice_out', 'double_play', 'field_error', 'fielders_choice', 'strikeout_double_play', 'walk', 'hit_by_pitch']) == True]
dr = dr[['game_date', 'batter', 'events', 'game_id']]
dr.head()

,game_date,batter,events,game_id
ind,,,,
116,2026-03-25,663757,strikeout,1
127,2026-03-25,592450,strikeout,1
134,2026-03-25,641355,field_out,1
144,2026-03-25,650333,walk,1
155,2026-03-25,656305,force_out,1


In [133]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=81)
hit_events = ['single', 'double', 'triple', 'home_run']

dr['had_hit'] = dr['events'].isin(hit_events).astype(int)

game_hits = (
    dr.groupby(['batter', 'game_id', 'game_date'])['had_hit']
    .max()
    .reset_index()
)
game_hits['game'] = 1
ps = game_hits[game_hits['game_date'] >= sev_days_ago][['batter', 'had_hit', 'game']]
ps = ps.groupby('batter', as_index = False).sum()
ps['hit_prob_s'] = ps['had_hit'] / ps['game']
ps = ps[['batter', 'hit_prob_s']]
ps.head()

,batter,hit_prob_s
0,457705,0.324324
1,467793,0.250000
2,500743,0.388889
3,502054,0.000000
4,502671,0.588235


In [134]:
fifteen_days_ago = latest - datetime.timedelta(days=10)

pf = game_hits[game_hits['game_date'] >= fifteen_days_ago][['batter', 'had_hit', 'game']]
pf = pf.groupby('batter', as_index = False).sum()
pf['hit_prob_f'] = pf['had_hit'] / pf['game']
pf = pf[['batter', 'hit_prob_f']]
pf.head()

,batter,hit_prob_f
0,457705,0.00
1,500743,0.25
2,502671,0.75
3,506702,0.00
4,516782,0.00


In [135]:
thirty_days_ago = latest - datetime.timedelta(days=30)

pt = game_hits[game_hits['game_date'] >= thirty_days_ago][['batter', 'had_hit', 'game']]
pt = pt.groupby('batter', as_index = False).sum()
pt['hit_prob_t'] = pt['had_hit'] / pt['game']
pt = pt[['batter', 'hit_prob_t']]
pt.head()

,batter,hit_prob_t
0,457705,0.312500
1,500743,0.294118
2,502671,0.695652
3,506702,0.166667
4,514888,0.750000


In [136]:
full = game_hits[['batter', 'had_hit', 'game']]
full = full.groupby('batter', as_index = False).sum()
full['hit_prob'] = full['had_hit'] / full['game']
full = full[['batter', 'hit_prob']]
full = full.merge(ps, on='batter', how='left').merge(pf, on='batter', how='left').merge(pt, on='batter', how='left').fillna(0)
full['Game_Hit_Probability'] = full['hit_prob'] * .175 + full['hit_prob_s'] * .225 + full['hit_prob_t'] * .275 + full['hit_prob_f'] * .325
full.head()

,batter,hit_prob,hit_prob_s,hit_prob_f,hit_prob_t,Game_Hit_Probability
0,457705,0.324324,0.324324,0.00,0.312500,0.215667
1,467793,0.250000,0.250000,0.00,0.000000,0.100000
2,500743,0.388889,0.388889,0.25,0.294118,0.317688
3,502054,0.000000,0.000000,0.00,0.000000,0.000000
4,502671,0.588235,0.588235,0.75,0.695652,0.670348


In [137]:
prob =full.rename(columns = {'batter' : 'key_mlbam'})[['key_mlbam', 'Game_Hit_Probability']]
prob.head()

,key_mlbam,Game_Hit_Probability
0,457705,0.215667
1,467793,0.100000
2,500743,0.317688
3,502054,0.000000
4,502671,0.670348


## WHOPS

In [138]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=7)

sev = dt[dt['game_date'] >= sev_days_ago]
def  ob(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 1
    elif df['events'] == 'triple':
        return 1
    elif df['events'] == 'home_run':
        return 1
    elif df['events'] == 'walk':
        return 1
    elif df['events'] == 'hit_by_pitch':
        return 1
    else:
        return 0

def  ab(df):
    if df['events'] == 'walk':
        return 0
    elif df['events'] == 'hit_by_pitch':
        return 0
    else:
        return 1

def  sv(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 2
    elif df['events'] == 'triple':
        return 3
    elif df['events'] == 'home_run':
        return 4
    else:
        return 0

sev['ob'] = sev.apply(ob, axis=1)
sev['sv'] = sev.apply(sv, axis=1)
sev['ab'] = sev.apply(ab, axis=1)
lsev = sev[sev['p_throws'] == "L"].drop('p_throws', axis=1)
lsev['pa_lsev'] = 1
lsev = lsev[['batter', 'ob', 'sv', 'ab', 'pa_lsev']].groupby('batter', as_index = False).sum()
lsev['lsev_slg'] = lsev['sv'] / lsev['pa_lsev']
lsev['lsev_obp'] = lsev['ob'] / lsev['pa_lsev']
lsev['lsev_ops'] = lsev['lsev_obp'] + lsev['lsev_slg']
lsev['lsev_rc'] = lsev['lsev_slg'] * lsev['lsev_obp'] * lsev['ab']/lsev['pa_lsev']
lsev = lsev[['batter', 'lsev_ops', 'pa_lsev', 'lsev_rc']]
lsev.head()

,batter,lsev_ops,pa_lsev,lsev_rc
0,500743,0.666667,9,0.076818
1,502671,1.125000,8,0.312500
2,506702,0.000000,4,0.000000
3,516782,0.166667,6,0.000000
4,518692,0.642857,14,0.102041


In [139]:
rsev = sev[sev['p_throws'] == "R"].drop('p_throws', axis=1)
rsev['pa_rsev'] = 1
rsev = rsev[['batter', 'ob', 'sv','pa_rsev', 'ab']].groupby('batter', as_index = False).sum()
rsev['rsev_slg'] = rsev['sv'] / rsev['pa_rsev']
rsev['rsev_obp'] = rsev['ob'] / rsev['pa_rsev']
rsev['rsev_ops'] = rsev['rsev_obp'] + rsev['rsev_slg']
rsev['rsev_rc'] = rsev['rsev_slg'] * rsev['rsev_obp'] * rsev['ab']/rsev['pa_rsev']
rsev = rsev[['batter', 'rsev_ops', 'pa_rsev', 'rsev_rc']]
rsev.head()

,batter,rsev_ops,pa_rsev,rsev_rc
0,457705,0.000000,3,0.000000
1,500743,1.000000,3,0.222222
2,502671,0.619048,21,0.086168
3,506702,0.000000,3,0.000000
4,516782,0.000000,1,0.000000


In [140]:
fif_days_ago = latest - datetime.timedelta(days=15)
fif = dt[dt['game_date'] >= fif_days_ago]
fif['ob'] = fif.apply(ob, axis=1)
fif['sv'] = fif.apply(sv, axis=1)
fif['ab'] = fif.apply(ab, axis=1)
lfif = fif[fif['p_throws'] == "L"].drop('p_throws', axis=1)
lfif['pa_lfif'] = 1
lfif = lfif[['batter', 'ob', 'sv','pa_lfif', 'ab']].groupby('batter', as_index = False).sum()
lfif['lfif_slg'] = lfif['sv'] / lfif['pa_lfif']
lfif['lfif_obp'] = lfif['ob'] / lfif['pa_lfif']
lfif['lfif_ops'] = lfif['lfif_obp'] + lfif['lfif_slg']
lfif['lfif_rc'] = lfif['lfif_slg'] * lfif['lfif_obp'] * lfif['ab']/lfif['pa_lfif']
lfif = lfif[['batter', 'lfif_ops', 'pa_lfif', 'lfif_rc']]
lfif.head()

,batter,lfif_ops,pa_lfif,lfif_rc
0,457705,0.416667,12,0.020833
1,500743,0.428571,14,0.034985
2,502671,1.400000,15,0.416000
3,506702,0.000000,8,0.000000
4,516782,0.400000,15,0.030815


In [141]:
rfif = fif[fif['p_throws'] == "R"].drop('p_throws', axis=1)
rfif['pa_rfif'] = 1
rfif = rfif[['batter', 'ob', 'sv','pa_rfif', 'ab']].groupby('batter', as_index = False).sum()
rfif['rfif_slg'] = rfif['sv'] / rfif['pa_rfif']
rfif['rfif_obp'] = rfif['ob'] / rfif['pa_rfif']
rfif['rfif_ops'] = rfif['rfif_obp'] + rfif['rfif_slg']
rfif['rfif_rc'] = rfif['rfif_slg'] * rfif['rfif_obp'] * rfif['ab']/rfif['pa_rfif']
rfif = rfif[['batter', 'rfif_ops', 'pa_rfif', 'rfif_rc']]
rfif.head()

,batter,rfif_ops,pa_rfif,rfif_rc
0,457705,0.000000,6,0.000000
1,500743,0.636364,11,0.081142
2,502671,0.405405,37,0.036484
3,506702,0.000000,10,0.000000
4,516782,0.000000,1,0.000000


In [142]:
thir_days_ago = latest - datetime.timedelta(days=30)
thir = dt[dt['game_date'] >= thir_days_ago]
thir['ob'] = thir.apply(ob, axis=1)
thir['sv'] = thir.apply(sv, axis=1)
thir['ab'] = thir.apply(ab, axis=1)
lthir = thir[thir['p_throws'] == "L"].drop('p_throws', axis=1)
lthir['pa_lthir'] = 1
lthir = lthir[['batter', 'ob', 'sv','pa_lthir', 'ab']].groupby('batter', as_index = False).sum()
lthir['lthir_slg'] = lthir['sv'] / lthir['pa_lthir']
lthir['lthir_obp'] = lthir['ob'] / lthir['pa_lthir']
lthir['lthir_ops'] = lthir['lthir_obp'] + lthir['lthir_slg']
lthir['lthir_rc'] = lthir['lthir_slg'] * lthir['lthir_obp'] * lthir['ab']/lthir['pa_lthir']
lthir = lthir[['batter', 'lthir_ops', 'pa_lthir', 'lthir_rc']]
lthir.head()

,batter,lthir_ops,pa_lthir,lthir_rc
0,457705,0.363636,22,0.020285
1,500743,0.440000,25,0.039424
2,502671,1.515152,33,0.475304
3,506702,0.266667,15,0.017778
4,514888,0.461538,13,0.049158


In [143]:
rthir = thir[thir['p_throws'] == "R"].drop('p_throws', axis=1)
rthir['pa_rthir'] = 1
rthir = rthir[['batter', 'ob', 'sv','pa_rthir', 'ab']].groupby('batter', as_index = False).sum()
rthir['rthir_slg'] = rthir['sv'] / rthir['pa_rthir']
rthir['rthir_obp'] = rthir['ob'] / rthir['pa_rthir']
rthir['rthir_ops'] = rthir['rthir_obp'] + rthir['rthir_slg']
rthir['rthir_rc'] = rthir['rthir_slg'] * rthir['rthir_obp'] * rthir['ab']/rthir['pa_rthir']
rthir = rthir[['batter', 'rthir_ops', 'pa_rthir', 'rthir_rc']]
rthir.head()

,batter,rthir_ops,pa_rthir,rthir_rc
0,457705,0.625000,16,0.082031
1,500743,0.550000,20,0.067500
2,502671,0.482143,56,0.051248
3,506702,0.000000,10,0.000000
4,514888,0.675676,37,0.106410


In [144]:
full = dt
full['ob'] = full.apply(ob, axis=1)
full['sv'] = full.apply(sv, axis=1)
full['ab'] = full.apply(ab, axis=1)
lfull = full[full['p_throws'] == "L"].drop('p_throws', axis=1)
lfull['pa_lfull'] = 1
lfull = lfull[['batter', 'ob', 'sv','pa_lfull', 'ab']].groupby('batter', as_index = False).sum()
lfull['lfull_slg'] = lfull['sv'] / lfull['pa_lfull']
lfull['lfull_obp'] = lfull['ob'] / lfull['pa_lfull']
lfull['lfull_ops'] = lfull['lfull_obp'] + lfull['lfull_slg']
lfull['lfull_rc'] = lfull['lfull_slg'] * lfull['lfull_obp'] * lfull['ab']/lfull['pa_lfull']
lfull = lfull[['batter', 'lfull_ops', 'pa_lfull', 'lfull_rc']]
lfull.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 518 entries, 0 to 517
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   batter     518 non-null    Int64  
 1   lfull_ops  518 non-null    float64
 2   pa_lfull   518 non-null    int64  
 3   lfull_rc   518 non-null    float64
dtypes: Int64(1), float64(2), int64(1)
memory usage: 16.8 KB


In [145]:
rfull = full[full['p_throws'] == "R"].drop('p_throws', axis=1)
rfull['pa_rfull'] = 1
rfull = rfull[['batter', 'ob', 'sv','pa_rfull', 'ab']].groupby('batter', as_index = False).sum()
rfull['rfull_slg'] = rfull['sv'] / rfull['pa_rfull']
rfull['rfull_obp'] = rfull['ob'] / rfull['pa_rfull']
rfull['rfull_ops'] = rfull['rfull_obp'] + rfull['rfull_slg']
rfull['rfull_rc'] = rfull['rfull_slg'] * rfull['rfull_obp'] * rfull['ab']/rfull['pa_rfull']
rfull = rfull[['batter', 'rfull_ops', 'pa_rfull', 'rfull_rc']]
rfull.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529 entries, 0 to 528
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   batter     529 non-null    Int64  
 1   rfull_ops  529 non-null    float64
 2   pa_rfull   529 non-null    int64  
 3   rfull_rc   529 non-null    float64
dtypes: Int64(1), float64(2), int64(1)
memory usage: 17.2 KB


In [146]:
whops = rfull.merge(lsev, on='batter', how='left').merge(lthir, on='batter', how='left').merge(lfif, on='batter', how='left').merge(lfull, on='batter', how='left').merge(rsev, on='batter', how='left').merge(rthir, on='batter', how='left').merge(rfif, on='batter', how='left')
from pybaseball import chadwick_register
names = chadwick_register()

whops =whops.rename(columns = {'batter' : 'key_mlbam'}).merge(names, on='key_mlbam', how='left')
whops = whops.drop(['key_mlbam', 'key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last'], axis=1)

whops[whops['name_last'] == 'Langford'].head()

Gathering player lookup table. This may take a moment.


,rfull_ops,pa_rfull,rfull_rc,lsev_ops,pa_lsev,lsev_rc,lthir_ops,pa_lthir,lthir_rc,lfif_ops,...,pa_rsev,rsev_rc,rthir_ops,pa_rthir,rthir_rc,rfif_ops,pa_rfif,rfif_rc,name_last,name_first
464,0.733333,60,0.127546,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Langford,Wyatt


In [147]:
whops['abt'] = whops['pa_lfull'] + whops['pa_rfull']
whops['abtl'] = whops['pa_lfull'] / whops['abt']
whops['abtr'] = whops['pa_rfull'] / whops['abt']
whops['whops_R'] = whops['rfull_ops']*.175 + whops['rsev_ops']*.225 + whops['rthir_ops']*.275 + whops['rfif_ops']*.325
whops['whops_L'] = whops['lfull_ops']*.175 + whops['lsev_ops']*.225 + whops['lthir_ops']*.275 + whops['lfif_ops']*.325
whops['whops'] = whops['whops_R']*whops['abtr'] + whops['whops_L']*whops['abtl']
whops['rcw_R'] = whops['rfull_rc']*.175 + whops['rsev_rc']*.225 + whops['rthir_rc']*.275 + whops['rfif_rc']*.325
whops['rcw_L'] = whops['lfull_rc']*.175 + whops['lsev_rc']*.225 + whops['lthir_rc']*.275 + whops['lfif_rc']*.325
whops['rcw'] = whops['rcw_R']*whops['abtr'] + whops['rcw_L']*whops['abtl']
whops['rope'] = whops['rcw'] + whops['whops']
WHOPS = whops[['name_first', 'name_last', 'whops', 'whops_R', 'whops_L', 'pa_lfull', 'pa_rfull', 'rcw_R', 'rcw_L', 'rcw', 'rope']]
WHOPS[(WHOPS['pa_lfull'] + WHOPS['pa_rfull']) > 30].nlargest(30, 'rope')

,name_first,name_last,whops,whops_R,whops_L,pa_lfull,pa_rfull,rcw_R,rcw_L,rcw,rope
383,Jesus,Rodriguez,1.422342,1.903333,0.312363,13.0,30,1.071347,0.026118,0.755348,2.177690
92,Jonah,Heim,1.144318,0.426063,1.900376,38.0,40,0.045649,1.049282,0.534598,1.678916
524,Jung Hoo,Lee,1.106730,1.242679,0.758793,59.0,151,0.414232,0.151386,0.340385,1.447115
99,Amed,Rosario,1.119597,1.054432,1.176943,50.0,44,0.261027,0.312232,0.288264,1.407861
54,Ketel,Marte,1.050892,0.767714,1.814614,66.0,178,0.137027,0.727219,0.296669,1.347561
139,Ronald,Acuña,1.072574,1.340531,0.581319,72.0,132,0.384007,0.054456,0.267695,1.340269
263,Luke,Raley,1.049918,1.114977,0.035000,10.0,156,0.269435,0.001750,0.253309,1.303227
185,Jeremy,Peña,0.975788,0.744854,2.089706,17.0,82,0.128623,1.141994,0.302637,1.278425
116,Yandy,Díaz,1.038632,1.054588,0.994626,62.0,171,0.234396,0.226666,0.232339,1.270971
349,Colton,Cowser,1.010611,1.086659,0.038889,9.0,115,0.263360,0.002160,0.244402,1.255013


In [148]:
WHOPS[WHOPS['name_last'] == "Rice" ].head()

,name_first,name_last,whops,whops_R,whops_L,pa_lfull,pa_rfull,rcw_R,rcw_L,rcw,rope
484,Ben,Rice,1.012069,1.036831,0.953675,67.0,158,0.228001,0.216779,0.224659,1.236728


# Weighted Bases

In [149]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=7)

sev = dt[dt['game_date'] >= sev_days_ago]

def  sv(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 2
    elif df['events'] == 'triple':
        return 3
    elif df['events'] == 'home_run':
        return 4
    else:
        return 0

sev['sv'] = sev.apply(sv, axis=1)
lsev = sev[sev['p_throws'] == "L"].drop('p_throws', axis=1)
lsev['pa_lsev'] = 1
lsev = lsev[['batter', 'sv', 'pa_lsev']].groupby('batter', as_index = False).sum()
lsev['lsev_slg'] = lsev['sv'] / lsev['pa_lsev']
lsev = lsev[['batter', 'lsev_slg', 'pa_lsev']]
lsev.head()

,batter,lsev_slg,pa_lsev
0,500743,0.222222,9
1,502671,0.625000,8
2,506702,0.000000,4
3,516782,0.000000,6
4,518692,0.357143,14


In [150]:
rsev = sev[sev['p_throws'] == "R"].drop('p_throws', axis=1)
rsev['pa_rsev'] = 1
rsev = rsev[['batter', 'sv','pa_rsev']].groupby('batter', as_index = False).sum()
rsev['rsev_slg'] = rsev['sv'] / rsev['pa_rsev']
rsev = rsev[['batter', 'rsev_slg', 'pa_rsev']]
rsev.head()

,batter,rsev_slg,pa_rsev
0,457705,0.000000,3
1,500743,0.666667,3
2,502671,0.333333,21
3,506702,0.000000,3
4,516782,0.000000,1


In [151]:
fif_days_ago = latest - datetime.timedelta(days=15)
fif = dt[dt['game_date'] >= fif_days_ago]
fif['sv'] = fif.apply(sv, axis=1)
lfif = fif[fif['p_throws'] == "L"].drop('p_throws', axis=1)
lfif['pa_lfif'] = 1
lfif = lfif[['batter', 'sv','pa_lfif']].groupby('batter', as_index = False).sum()
lfif['lfif_slg'] = lfif['sv'] / lfif['pa_lfif']
lfif = lfif[['batter', 'lfif_slg', 'pa_lfif']]
lfif.head()

,batter,lfif_slg,pa_lfif
0,457705,0.083333,12
1,500743,0.142857,14
2,502671,0.800000,15
3,506702,0.000000,8
4,516782,0.133333,15


In [152]:
rfif = fif[fif['p_throws'] == "R"].drop('p_throws', axis=1)
rfif['pa_rfif'] = 1
rfif = rfif[['batter', 'sv','pa_rfif']].groupby('batter', as_index = False).sum()
rfif['rfif_slg'] = rfif['sv'] / rfif['pa_rfif']
rfif = rfif[['batter', 'rfif_slg', 'pa_rfif']]
rfif.head()

,batter,rfif_slg,pa_rfif
0,457705,0.000000,6
1,500743,0.272727,11
2,502671,0.189189,37
3,506702,0.000000,10
4,516782,0.000000,1


In [153]:
thir_days_ago = latest - datetime.timedelta(days=30)
thir = dt[dt['game_date'] >= thir_days_ago]
thir['sv'] = thir.apply(sv, axis=1)
lthir = thir[thir['p_throws'] == "L"].drop('p_throws', axis=1)
lthir['pa_lthir'] = 1
lthir = lthir[['batter', 'sv', 'pa_lthir']].groupby('batter', as_index = False).sum()
lthir['lthir_slg'] = lthir['sv'] / lthir['pa_lthir']
lthir = lthir[['batter', 'lthir_slg', 'pa_lthir']]
lthir.head()

,batter,lthir_slg,pa_lthir
0,457705,0.090909,22
1,500743,0.160000,25
2,502671,0.939394,33
3,506702,0.133333,15
4,514888,0.230769,13


In [154]:
rthir = thir[thir['p_throws'] == "R"].drop('p_throws', axis=1)
rthir['pa_rthir'] = 1
rthir = rthir[['batter', 'ob', 'sv','pa_rthir', 'ab']].groupby('batter', as_index = False).sum()
rthir['rthir_slg'] = rthir['sv'] / rthir['pa_rthir']
rthir = rthir[['batter', 'rthir_slg', 'pa_rthir']]
rthir.head()

,batter,rthir_slg,pa_rthir
0,457705,0.250000,16
1,500743,0.250000,20
2,502671,0.214286,56
3,506702,0.000000,10
4,514888,0.378378,37


In [155]:
full = dt
full['sv'] = full.apply(sv, axis=1)
lfull = full[full['p_throws'] == "L"].drop('p_throws', axis=1)
lfull['pa_lfull'] = 1
lfull = lfull[['batter', 'ob', 'sv','pa_lfull', 'ab']].groupby('batter', as_index = False).sum()
lfull['lfull_slg'] = lfull['sv'] / lfull['pa_lfull']
lfull = lfull[['batter', 'lfull_slg', 'pa_lfull']]
lfull.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 518 entries, 0 to 517
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   batter     518 non-null    Int64  
 1   lfull_slg  518 non-null    float64
 2   pa_lfull   518 non-null    int64  
dtypes: Int64(1), float64(1), int64(1)
memory usage: 12.8 KB


In [156]:
rfull = full[full['p_throws'] == "R"].drop('p_throws', axis=1)
rfull['pa_rfull'] = 1
rfull = rfull[['batter', 'sv','pa_rfull']].groupby('batter', as_index = False).sum()
rfull['rfull_slg'] = rfull['sv'] / rfull['pa_rfull']
rfull = rfull[['batter', 'rfull_slg', 'pa_rfull']]
rfull.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529 entries, 0 to 528
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   batter     529 non-null    Int64  
 1   rfull_slg  529 non-null    float64
 2   pa_rfull   529 non-null    int64  
dtypes: Int64(1), float64(1), int64(1)
memory usage: 13.0 KB


In [157]:
wslg = rfull.merge(lsev, on='batter', how='left').merge(lthir, on='batter', how='left').merge(lfif, on='batter', how='left').merge(lfull, on='batter', how='left').merge(rsev, on='batter', how='left').merge(rthir, on='batter', how='left').merge(rfif, on='batter', how='left')
from pybaseball import chadwick_register
names = chadwick_register()

wslg =wslg.rename(columns = {'batter' : 'key_mlbam'}).merge(names, on='key_mlbam', how='left')
wslg = wslg.drop(['key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last'], axis=1)

wslg[wslg['name_last'] == 'Langford'].head()

Gathering player lookup table. This may take a moment.


,key_mlbam,rfull_slg,pa_rfull,lsev_slg,pa_lsev,lthir_slg,pa_lthir,lfif_slg,pa_lfif,lfull_slg,pa_lfull,rsev_slg,pa_rsev,rthir_slg,pa_rthir,rfif_slg,pa_rfif,name_last,name_first
464,694671,0.416667,60,NaN,NaN,NaN,NaN,NaN,NaN,0.166667,24.0,NaN,NaN,NaN,NaN,NaN,NaN,Langford,Wyatt


In [158]:
wslg['ab'] = wslg['pa_lfull'] + wslg['pa_rfull']
wslg['abr'] = wslg['pa_rfull'] / wslg['ab']
wslg['abl'] = wslg['pa_lfull'] / wslg['ab']
wslg['WTB_r'] = wslg['rsev_slg']*.225 + wslg['rthir_slg']*.275 + wslg['rfif_slg']*.325 + wslg['rfull_slg']*.175
wslg['WTB_l'] = wslg['lsev_slg']*.225 + wslg['lthir_slg']*.275 + wslg['lfif_slg']*.325 + wslg['lfull_slg']*.175
wslg['WTB'] = wslg['WTB_r']*wslg['abr'] + wslg['WTB_l']*wslg['abl']
wslg['Expected_Bases'] = wslg['WTB']*3.5
WTB = wslg[['name_first', 'name_last', 'pa_lfull', 'pa_rfull', 'WTB_l', 'WTB_r', 'WTB', 'Expected_Bases', 'key_mlbam']]
WTB[WTB['pa_rfull'] + WTB['pa_lfull'] > 70].nlargest(30, 'Expected_Bases')

,name_first,name_last,pa_lfull,pa_rfull,WTB_l,WTB_r,WTB,Expected_Bases,key_mlbam
92,Jonah,Heim,38.0,40,1.468399,0.186625,0.811079,2.838777,641680
99,Amed,Rosario,50.0,44,0.798794,0.738393,0.770521,2.696824,642708
54,Ketel,Marte,66.0,178,1.336859,0.405332,0.657302,2.300558,606466
263,Luke,Raley,10.0,156,0.017500,0.696081,0.655203,2.293209,670042
139,Ronald,Acuña,72.0,132,0.209167,0.851853,0.625023,2.187580,660670
116,Yandy,Díaz,62.0,171,0.580818,0.639592,0.623953,2.183834,650490
189,Juan,Soto,69.0,108,0.596290,0.633988,0.619292,2.167523,665742
524,Jung Hoo,Lee,59.0,151,0.374332,0.713255,0.618034,2.163118,808982
484,Ben,Rice,67.0,158,0.539361,0.645151,0.613649,2.147771,700250
266,Yordan,Álvarez,73.0,179,0.945748,0.467389,0.605961,2.120865,670541


In [159]:
WAVE = WAVE.drop(['name_first', 'name_last'], axis = 1)
WAVE = WAVE.merge(WTB, on='key_mlbam', how ='left').merge(prob, on = 'key_mlbam', how = 'left')
WAVE['Consistency'] = WAVE['Game_Hit_Probability'] - WAVE['probability']
WAVE['Approach'] = WAVE['Game_Hit_Probability'] * WAVE['probability']
WAVE = WAVE[['name_first', 'name_last', 'pa_lfull', 'pa_rfull', 'probability_L', 'probability_R', 'probability', 'Game_Hit_Probability', 'Consistency', 'Approach', 'Expected_Bases']].fillna(0)
WAVE = WAVE[WAVE['Consistency'] > 0]
WAVE = WAVE.sort_values('Game_Hit_Probability', ascending=False)
WAVE = WAVE.rename(columns = {'pa_lfull' : 'PA_L', 'pa_rfull' : 'PA_R'})
pa = WAVE['PA_L'] + WAVE['PA_R']
wall = pa.max() * .25
WAVE = WAVE[(WAVE['PA_L'] + WAVE['PA_R']) > wall]

# Look Ups

In [54]:
WAVE[(WAVE['PA_L'] + WAVE['PA_R']) > 130].nlargest(30, 'Game_Hit_Probability')

,name_first,name_last,PA_L,PA_R,probability_L,probability_R,probability,Game_Hit_Probability,Consistency,Approach,Expected_Bases
59,Alex,Bregman,67.0,200,0.554392,0.643304,0.622310,0.847379,0.225070,0.527332,1.165531
434,Junior,Caminero,70.0,179,0.671679,0.624044,0.637903,0.826794,0.188891,0.527414,1.614401
303,Ernie,Clement,64.0,171,0.514235,0.758789,0.703926,0.794013,0.090087,0.558927,1.885553
223,Spencer,Steer,56.0,170,0.680469,0.605780,0.625339,0.791959,0.166621,0.495242,1.166955
286,Maikel,García,48.0,196,0.862317,0.637464,0.694285,0.781628,0.087342,0.542673,1.157394
322,Bobby,Witt,52.0,208,0.667310,0.620668,0.630348,0.773393,0.143045,0.487507,1.548563
517,Luke,Keaschall,73.0,151,0.697620,0.636982,0.657616,0.762054,0.104437,0.501139,1.246549
510,Chase,Meidroth,62.0,178,0.770297,0.576494,0.634789,0.761814,0.127026,0.483591,1.449011
249,Xavier,Edwards,58.0,200,0.634611,0.662505,0.656372,0.760358,0.103986,0.499077,1.464539
412,Spencer,Horwitz,31.0,173,0.635427,0.674288,0.668583,0.760091,0.091508,0.508184,1.771934


In [55]:
name = 'Antonacci'
WAVE[(WAVE['name_last'] == name) | (WAVE['name_first'] == name)].head().sort_values(by='probability', ascending=False)

,name_first,name_last,PA_L,PA_R,probability_L,probability_R,probability,Game_Hit_Probability,Consistency,Approach,Expected_Bases


In [56]:
pave[pave['at_bats'] >150].nlargest(30, 'PAVE_PLUS')

,name_first,name_last,at_bats,PAVE_PLUS,Expected_Hits,Expected_Bases,Expected_HRs
22,Michael,Lorenzen,275,1.389548,7.278097,11.366238,0.840368
84,Kyle,Freeland,228,1.331207,6.913418,12.540975,1.500774
323,Matthew,Liberatore,264,1.288887,5.808205,8.354628,0.617051
179,Jack,Flaherty,242,1.255874,5.558522,9.748859,0.765034
245,Chris,Paddack,212,1.235029,6.541405,9.054366,0.622399
582,Landen,Roupp,269,1.234610,5.631177,8.534555,0.283733
427,Simeon,Woods Richardson,224,1.225907,6.111790,9.428919,0.559576
469,Jacob,Lopez,237,1.216761,6.129775,10.175191,0.970853
144,Tyler,Mahle,244,1.214936,5.810574,10.016197,1.010656
240,Brady,Singer,236,1.201521,6.235975,12.980621,1.952056


In [57]:
name = 'Flaherty'
pave[(pave['name_last'] == name) | (pave['name_first'] == name)].head()

,name_first,name_last,at_bats,PAVE_PLUS,Expected_Hits,Expected_Bases,Expected_HRs
179,Jack,Flaherty,242,1.255874,5.558522,9.748859,0.765034


#Strength

In [58]:
gam_id = df[["game_date", "home_team", "away_team", "pitcher", "at_bat_number"]].drop_duplicates().iloc[::-1]

def create_id(df, col_check):
  count = 0
  new_col = []
  for val in df[col_check]:
      if val == 1:
          count += 1
      new_col.append(count)
  df['game_id'] = new_col
  return df

gam_id = create_id(gam_id, 'at_bat_number')

data = df.merge(gam_id, on = ["game_date", "home_team", "away_team", "pitcher", "at_bat_number"])
data['ind'] = (data['game_id'].astype('str') + data['at_bat_number'].astype('str') + data['pitch_number'].astype('str')).astype('int')
data = data.set_index('ind')
data = data.sort_index()

In [59]:
hhr = data[data['inning_topbot'] == 'Bot'][['home_team', 'events', 'home_score', 'post_home_score', 'game_id']]
hhr = hhr.rename(columns = {'home_team' : 'team', 'home_score' : 'pre_score', 'post_home_score' : 'post_score'})
ahr = data[data['inning_topbot'] == 'Top'][['away_team', 'events', 'away_score', 'post_away_score', 'game_id']]
ahr = ahr.rename(columns = {'away_team' : 'team', 'away_score' : 'pre_score', 'post_away_score' : 'post_score'})
hr = pd.concat([hhr, ahr])

def homer(df):
  if df['events'] == 'home_run':
    return 'homer'
  else:
    return 'non_homer'

hr['hr'] = hr.apply(homer, axis = 1)

gp = hr[['team', 'game_id']].drop_duplicates()
gp = gp.groupby('team', as_index = False).count()
gp = gp.rename(columns = {'game_id' : 'game_count'})
total_homers = hr[['team', 'hr']][hr['hr'] == 'homer'].groupby('team', as_index = False).count()
hpg = total_homers.merge(gp, on = 'team')
hpg['homer_per_game'] = hpg['hr'] / hpg['game_count']
hpg = hpg[['team', 'homer_per_game']]
gwh = hr[['team', 'game_id', 'hr']][hr['hr'] == 'homer'].groupby(['team', 'game_id'], as_index = False).count()
game = hr[['team', 'game_id']].drop_duplicates()
gwh = game.merge(gwh, on = ['team', 'game_id'], how = 'left').fillna(0)
ghi = gwh[gwh['hr'] > 0].groupby('team', as_index = False).count()
ghi = ghi.rename(columns = {'hr' : 'games_homered_in'})
gnhi = gwh[gwh['hr'] == 0].groupby('team', as_index = False).count()
gnhi = gnhi.rename(columns = {'hr' : 'games_not_homered_in'})
ghi = ghi.merge(gnhi, on = 'team', how = 'left')
ghi['game_homer_rate'] = ghi['games_homered_in'] / (ghi['games_homered_in'] + ghi['games_not_homered_in'])
ghi = ghi[['team', 'game_homer_rate']]
hr = hr.fillna(0)
hr = hr[hr['pre_score'] != hr['post_score']][['team', 'hr', 'pre_score', 'post_score']]
hr = hr.groupby(['team', 'hr'], as_index = False).sum()
hr['runs_scored'] = hr['post_score'] - hr['pre_score']
hr = hr.pivot(index = 'team', columns = 'hr', values = 'runs_scored')
hr = hr.fillna(0).reset_index()
hr['runs_scored'] = hr['homer'] + hr['non_homer']
hr['home_run_reliance'] = hr['homer'] / hr['runs_scored']
hr['percent_on_non_homer'] = hr['non_homer'] / hr['runs_scored']
hr = hr.merge(hpg, on = 'team')
hr = hr.merge(ghi, on = 'team')
hr = hr[['team', 'runs_scored', 'home_run_reliance', 'percent_on_non_homer', 'homer_per_game', 'game_homer_rate']]
for_merge = hr[['team', 'home_run_reliance', 'homer_per_game', 'game_homer_rate']]

In [60]:
sus = data[['home_team', 'events']]
sus = sus.rename(columns = {'home_team' : 'team'})
sus['hr'] = sus.apply(homer, axis = 1)
sus = sus[sus['hr'] == 'homer']
sus = sus[['team', 'hr']]
sus = sus.groupby('team').count().reset_index()
games = data[['home_team', 'game_id']].drop_duplicates()
games = games.groupby('home_team').count().reset_index()
games = games.rename(columns = {'home_team' : 'team', 'game_id' : 'game_count'})
asus = data[['home_team', 'events', 'inning_topbot']]
asus = asus[asus['inning_topbot'] == 'Top']
asus['hr'] = asus.apply(homer, axis = 1)
asus = asus[asus['hr'] == 'homer']
asus = asus.rename(columns = {'home_team' : 'team', 'hr' : 'away_hr'})
asus = asus[['team', 'away_hr']]
asus = asus.groupby('team').count().reset_index()
sus = sus.merge(games, on = 'team')
sus['hr_rate'] = sus['hr'] / sus['game_count']
sus = sus.merge(asus, on = 'team')
sus['away_hr_rate'] = sus['away_hr'] / sus['game_count']
sus['team_home_run_rate'] = sus['hr_rate'] - sus['away_hr_rate']
sus = sus[['team', 'team_home_run_rate', 'away_hr_rate']]

In [61]:
def  sv(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 2
    elif df['events'] == 'triple':
        return 3
    elif df['events'] == 'home_run':
        return 4
    elif df['events'] == 'walk':
        return 1
    else:
        return 0
top = data[data['inning_topbot'] == 'Top'][['game_id', 'away_team', 'events', 'home_team']]
top = top.rename(columns = {'away_team' : 'team', 'home_team' : 'opp'})
bot = data[data['inning_topbot'] == 'Bot'][['game_id', 'home_team', 'events', 'away_team']]
bot = bot.rename(columns = {'home_team' : 'team', 'away_team' : 'opp'})
bases = pd.concat([top, bot])
bases['bases'] = bases.apply(sv, axis = 1)
bases = bases.groupby(['team', 'opp', 'game_id'], as_index = False)['bases'].sum()
ba = bases.rename(columns = {'opp' : 'team', 'team' : 'opp', 'bases' : 'bases_allowed'})
full_bases = bases.merge(ba, on = ['team', 'opp','game_id'])

full_bases = full_bases.sort_values(['team', 'game_id'])

full_bases['bases_shift'] = (
    full_bases.groupby('team')['bases']
      .shift(1)
)

full_bases['bases_allowed_shift'] = (
    full_bases.groupby('team')['bases_allowed']
      .shift(1)
)

windows = [10, 30, 81]

for w in windows:

    # offensive rolling bases
    full_bases[f'bases_pg_{w}'] = (
        full_bases.groupby('team')['bases_shift']
          .rolling(w, min_periods=1)
          .mean()
          .reset_index(level=0, drop=True)
    )

    # defensive rolling bases allowed
    full_bases[f'bases_allowed_pg_{w}'] = (
        full_bases.groupby('team')['bases_allowed_shift']
          .rolling(w, min_periods=1)
          .mean()
          .reset_index(level=0, drop=True)
    )


full_bases['bases_pg_full'] = (
    full_bases.groupby('team')['bases_shift']
      .expanding()
      .mean()
      .reset_index(level=0, drop=True)
)

full_bases['bases_allowed_pg_full'] = (
    full_bases.groupby('team')['bases_allowed_shift']
      .expanding()
      .mean()
      .reset_index(level=0, drop=True)
)

full_bases['WAVE'] = (
    0.3 * full_bases['bases_pg_10'] +
    0.3 * full_bases['bases_pg_30'] +
    0.2 * full_bases['bases_pg_81'] +
    0.2 * full_bases['bases_pg_full']
)

full_bases['WAVE_allowed'] = (
    0.3 * full_bases['bases_allowed_pg_10'] +
    0.3 * full_bases['bases_allowed_pg_30'] +
    0.2 * full_bases['bases_allowed_pg_81'] +
    0.2 * full_bases['bases_allowed_pg_full']
)

opp_wave = full_bases[['team', 'game_id', 'WAVE_allowed']].rename(columns={
    'team': 'opp',
    'WAVE_allowed': 'opp_WAVE_allowed'
})

full_bases = full_bases.merge(
    opp_wave,
    on=['opp', 'game_id'],
    how='left'
)

full_bases = full_bases.fillna(0)

full_bases['offensive_edge'] = (
    full_bases['WAVE']
    -
    full_bases['opp_WAVE_allowed']
)

latest = full_bases.groupby('team', as_index = False)['game_id'].max()
latest_off = latest.merge(full_bases, on = ['team', 'game_id'])[['team', 'offensive_edge']]

In [62]:
dt = data[['home_team', 'away_team', 'game_date', 'game_id', 'post_home_score', 'post_away_score']]
dt['fs'] = dt['post_home_score'] + dt['post_away_score']
gf = dt.groupby('game_id', as_index = False)['fs'].max()
gf = gf.merge(dt, on = ['game_id', 'fs']).drop_duplicates()[['home_team', 'away_team', 'game_date', 'game_id', 'post_home_score', 'post_away_score']]

In [63]:
away = gf.rename(columns = {'away_team' : 'team', 'home_team' : 'opp', 'post_home_score' : 'ra', 'post_away_score' : 'rs'})
home = gf.rename(columns = {'home_team' : 'team', 'away_team' : 'opp', 'post_home_score' : 'rs', 'post_away_score' : 'ra'})
record = pd.concat([away, home]).sort_values(['team', 'game_date'])
record['win'] = (record['rs'] > record['ra']).astype('int')
record['loss'] = (record['rs'] < record['ra']).astype('int')
record = record[['opp', 'team', 'game_date', 'game_id', 'win', 'loss', 'rs', 'ra']]

In [64]:
under3 = record.sort_values(['team', 'game_id'])
under3['under3'] = (under3['rs'] < 3).astype(int)
windows = [10, 30, 81, 162]

for w in windows:

    under3[f'under3_pct_{w}'] = (
        under3.groupby('team')['under3']
              .rolling(w, min_periods=1)
              .mean()
              .reset_index(level=0, drop=True)
    )

max3 = under3.groupby('team', as_index = False)['game_id'].max()
max3 = max3.merge(under3, on = ['team', 'game_id'])
max3['suppression_resistance'] = 1 - (max3['under3_pct_162'] * .2 + max3['under3_pct_81'] * .2 + max3['under3_pct_30'] * .3 + max3['under3_pct_10'] * .3)
max3 = max3[['team', 'suppression_resistance']]
sup_mean = max3['suppression_resistance'].mean()
std_sup = max3['suppression_resistance'].std()
max3['suppression_resistance'] = 1 + (((max3['suppression_resistance'] - sup_mean) / std_sup) * .15)

In [65]:
import pandas as pd
import numpy as np

nf = record.sort_values(['team', 'game_id']).reset_index(drop=True)

windows = [10, 30, 81]

def compute_windows(group):
    group = group.copy()

    results = {f'roll_{w}_excl': [] for w in windows}
    full_excl = []

    for i in range(len(group)):
        opp = group.iloc[i]['opp']

        history = group.iloc[:i]

        # exclude games vs this opponent
        history_excl = history[history['opp'] != opp]

        # full record
        if len(history_excl) > 0:
            full_excl.append(history_excl['win'].mean())
        else:
            full_excl.append(np.nan)

        # rolling windows
        for w in windows:
            if len(history_excl) > 0:
                results[f'roll_{w}_excl'].append(
                    history_excl['win'].tail(w).mean()
                )
            else:
                results[f'roll_{w}_excl'].append(np.nan)

    for k in results:
        group[k] = results[k]

    group['full_excl'] = full_excl

    return group

nf = nf.groupby('team', group_keys=False).apply(compute_windows)
nf = nf.rename(columns = {'roll_10_excl' : 'rolling_10', 'roll_30_excl' : 'rolling_30', 'roll_81_excl' : 'rolling_81', 'full_excl' : 'full'}).fillna(0)

In [66]:
def compute_current(group):
    group = group.copy()

    results = {f'roll_{w}_cur': [] for w in windows}
    full_excl = []

    for i in range(len(group)):
        opp = group.iloc[i]['opp']

        history = group.iloc[:i]

        if len(history) > 0:
            full_excl.append(history['win'].mean())
        else:
            full_excl.append(np.nan)

        for w in windows:
            if len(history) > 0:
                results[f'roll_{w}_cur'].append(
                    history['win'].tail(w).mean()
                )
            else:
                results[f'roll_{w}_cur'].append(np.nan)

    for k in results:
        group[k] = results[k]

    group['full_cur'] = full_excl

    return group

nf = nf.groupby('team', group_keys=False).apply(compute_current)
nf = nf.fillna(0)

In [67]:
nf['rs_shift'] = nf.groupby('team')['rs'].shift(1)
nf['ra_shift'] = nf.groupby('team')['ra'].shift(1)
windows = [10, 30, 81]

for w in windows:

    nf[f'rs_{w}'] = (
        nf.groupby('team')['rs_shift']
          .rolling(w, min_periods=1)
          .sum()
          .reset_index(level=0, drop=True)
    )

    nf[f'ra_{w}'] = (
        nf.groupby('team')['ra_shift']
          .rolling(w, min_periods=1)
          .sum()
          .reset_index(level=0, drop=True)
    )

nf['rs_full'] = (
    nf.groupby('team')['rs_shift']
      .cumsum()
)

nf['ra_full'] = (
    nf.groupby('team')['ra_shift']
      .cumsum()
)

EXP = 1.83

for w in windows:

    nf[f'pyth_{w}'] = (
        nf[f'rs_{w}'] ** EXP
    ) / (
        (nf[f'rs_{w}'] ** EXP) +
        (nf[f'ra_{w}'] ** EXP)
    )

nf['pyth_full'] = (
    nf['rs_full'] ** EXP
) / (
    (nf['rs_full'] ** EXP) +
    (nf['ra_full'] ** EXP)
)

nf = nf.fillna(0)

In [68]:
from numpy._core.fromnumeric import std
nf['strength'] = (nf['rolling_10']*.35 + nf['rolling_30']*.3 + nf['rolling_81']*.2 + nf['full']*.15)
nf['current_strength'] = nf['roll_10_cur']*.35 + nf['roll_30_cur']*.3 + nf['roll_81_cur']*.2 + nf['full_cur']*.15
nf['pyth_strength'] = (nf['pyth_10']*.35 + nf['pyth_30']*.3 + nf['pyth_81']*.2 + nf['pyth_full']*.15)
sos = nf.groupby('opp', as_index = False)[['strength', 'pyth_strength']].mean()
sos = sos.rename(columns = {'opp' : 'team', 'strength' : 'SOS', 'pyth_strength' : 'pyth_SOS'})
latest = nf.groupby('team', as_index = False)['game_id'].max()
current_strength = latest.merge(nf, on = ['team', 'game_id'])[['team', 'strength', 'current_strength', 'pyth_strength']]
master = current_strength.merge(sos, on = 'team')
mean_strength = master['strength'].mean()
mean_SOS = master['SOS'].mean()
mean_current_strength = master['current_strength'].mean()
mean_pyth_strength = master['pyth_strength'].mean()
mean_pyth_SOS = master['pyth_SOS'].mean()
std_strength = master['strength'].std()
std_SOS = master['SOS'].std()
std_current_strength = master['current_strength'].std()
std_pyth_strength = master['pyth_strength'].std()
std_pyth_SOS = master['pyth_SOS'].std()
master['norm_strength'] = 1 + (((master['strength'] - mean_strength) / std_strength) * .15)
master['norm_sos'] = 1 + (((master['SOS'] - mean_SOS) / std_SOS) * .15)
master['norm_current_strength'] = 1 + (((master['current_strength'] - mean_current_strength) / std_current_strength) * .15)
master['norm_pyth_strength'] = 1 + (((master['pyth_strength'] - mean_pyth_strength) / std_pyth_strength) * .15)
master['norm_pyth_sos'] = 1 + (((master['pyth_SOS'] - mean_pyth_SOS) / std_pyth_SOS) * .15)
master = master[['team', 'norm_strength', 'norm_sos', 'norm_current_strength', 'norm_pyth_strength', 'norm_pyth_sos']].rename(columns = {'norm_strength' : 'Strength', 'norm_sos' : 'SOS', 'norm_current_strength' : 'current', 'norm_pyth_strength' : 'pyth_Strength', 'norm_pyth_sos' : 'pyth_SOS'}).drop_duplicates().reset_index(drop = True)
master['Confidence'] = master['Strength'] + (master['SOS'] *.3)
master['pyth_Confidence'] = master['pyth_Strength'] + (master['pyth_SOS'] *.3)
mean_conf = master['Confidence'].mean()
std_conf = master['Confidence'].std()
mean_pyth_conf = master['pyth_Confidence'].mean()
std_pyth_conf = master['pyth_Confidence'].std()
master['Confidence'] = 1 + (((master['Confidence'] - mean_conf)/std_conf)*.15)
master['pyth_Confidence'] = 1 + (((master['pyth_Confidence'] - mean_pyth_conf) / std_pyth_conf) * .15)
master = master[['team', 'current', 'Strength', 'pyth_Strength', 'SOS', 'pyth_SOS', 'Confidence', 'pyth_Confidence']]
master['Confidence_Delta'] = master['Confidence'] - master['pyth_Confidence']
master = master.merge(latest_off, on = 'team')
master = master.merge(for_merge, on = 'team')
master = master.merge(sus, on = 'team')
master = master.merge(max3, on = 'team')
mean_edge = master['offensive_edge'].mean()
std_edge = master['offensive_edge'].std()
master['offensive_edge'] = 1 + (((master['offensive_edge'] - mean_edge) / std_edge) * .15)
master = master.drop_duplicates()
master['true_power'] = (master['offensive_edge'] + master['suppression_resistance'])/2
master = master[['team', 'current', 'Strength', 'pyth_Strength', 'SOS', 'pyth_SOS', 'Confidence', 'pyth_Confidence', 'Confidence_Delta', 'true_power', 'offensive_edge', 'suppression_resistance', 'home_run_reliance', 'homer_per_game', 'game_homer_rate', 'team_home_run_rate', 'away_hr_rate']]
master = master.sort_values('pyth_Confidence', ascending = False).reset_index(drop = True)

#GITHUB

In [162]:
repo = "/content/mlb_metrics"

WAVE.to_csv(
    "/data/wave.csv",
    index=False
)

pave.to_csv(
    "/data/pave.csv",
    index=False
)

master.to_csv(
    "/data/confidence.csv",
    index=False
)